In [ ]:
import pandas as pd
import re

# Carregar a planilha
df = pd.read_excel('Capina_Corrigido.xlsx', sheet_name='Sheet1')

# Função para extrair todas as doses de uma célula
def extrair_doses(texto):
    if not isinstance(texto, str):
        return {}
    # Encontra tudo no formato: Nome - (número)
    padrao = r'([A-Za-zÀ-ÿ\s]+?)\s*-\s*\(?([\d,]+)\)?'
    matches = re.findall(padrao, texto)
    doses = {}
    for nome, dose_str in matches:
        nome = nome.strip().replace(' (Transorb R)', '').replace(' (Herburon)', '')
        dose = float(dose_str.replace(',', '.'))
        doses[nome] = dose
    return doses

# Aplicar a extração
df['doses_dict'] = df['INSUMO_DOSE'].apply(extrair_doses)

# Dose máxima permitida
dose_max = {
    'Hexazinona': 2.00, 'Provence': 0.35, 'Ametrina': 8.00, 'Combine': 2.40,
    'Glifosato': 4.50, 'Transorb': 4.50, 'Diuron': 6.40, 'Herburon': 6.40,
    'Mesotriona': 0.50, 'Agridex': 2.00, 'Veloce': 2.00, 'Contain': 2.00,
    'Flumyzin': 0.30, 'Boral': 1.60, 'Triclon': 0.80, 'Alion': 2.00, 'Fipronil': 0.25
}

# Verificar excessos
excedidos = []
for idx, row in df.iterrows():
    for insumo, dose in row['doses_dict'].items():
        insumo_norm = insumo.strip()
        if insumo_norm in dose_max and dose > dose_max[insumo_norm] + 0.001:  # tolerância pequena
            excedidos.append({
                'Linha': idx+2,
                'CODIGO': row['CODIGO'],
                'Insumo': insumo_norm,
                'Dose_Usada': dose,
                'Dose_Max': dose_max[insumo_norm]
            })

# Mostrar resultados
erros = pd.DataFrame(excedidos)
print(f"Total de aplicações com dose acima do permitido: {len(erros)}")
if not erros.empty:
    print(erros)
else:
    print("✅ Todas as doses estão dentro do limite!")

In [ ]:
import pandas as pd
import re

df = pd.read_excel('Capina_Julho_Agosto.xlsx', sheet_name='Planilha2')

dose_max = {
    'Hexazinona': 2.00, 'Triclon': 0.80, 'Glifosato': 4.50, 'Transorb': 4.50,
    'Flumyzin': 0.30, 'Boral': 1.60, 'Alion': 2.00, 'Diuron': 6.40,
    'Herburon': 6.40, 'Ametrina': 8.00, 'Mesotriona': 0.50, 'Veloce': 10.00,
    'Spike': 2.40, 'Contain': 2.00, 'Provence': 0.35, 'Agridex': 2.00,
    'Fipronil': 0.25, 'Flumizyn': 0.30
}

def extrair_doses(texto):
    if not isinstance(texto, str):
        return {}
    padrao = r'([A-Za-zÀ-ÿ\s()]+?)\s*-\s*\(?([\d,]+)\)?'
    matches = re.findall(padrao, texto, re.IGNORECASE)
    doses = {}
    for nome, dose_str in matches:
        nome_clean = nome.strip().replace(' (Transorb R)', '').replace(' (Herburon)', '').strip()
        dose = float(dose_str.replace(',', '.'))
        doses[nome_clean] = dose
    return doses

# ================== 0. CORRIGIR ERRO DE VÍRGULA (ex: 24 → 2,4) ==================
# Se o valor for >= 10× o máximo permitido, move a vírgula uma casa (divide por 10)
correcoes_virgula = []

for idx in range(len(df)):
    texto = str(df.at[idx, 'INSUMO_DOSE'])
    if not texto or texto == 'nan':
        continue

    doses = extrair_doses(texto)
    novo_texto = texto

    for insumo, dose_usada in doses.items():
        if insumo in dose_max:
            maximo = dose_max[insumo]
            # Erro de vírgula: valor é >= 10x o máximo (ex: 24 quando max é 2.4)
            if dose_usada >= maximo * 10 - 0.001:
                dose_corrigida = round(dose_usada / 10, 3)
                padrao_sub = re.compile(
                    re.escape(insumo) + r'(.*?-\s*\(?)' + re.escape(str(dose_usada).replace('.', ',')).rstrip('0').rstrip(',') + r'(\)?)',
                    re.IGNORECASE
                )
                # Substituição robusta: busca o número exato no texto
                padrao_num = re.compile(
                    re.escape(insumo) + r'(.*?-\s*\(?)(\d[\d,]*)(\)?)',
                    re.IGNORECASE
                )
                def replace_dose(m):
                    val_str = m.group(2).replace(',', '.')
                    try:
                        val = float(val_str)
                    except:
                        return m.group(0)
                    if abs(val - dose_usada) < 0.001:
                        return f"{insumo}{m.group(1)}{dose_corrigida:.3f}{m.group(3)}"
                    return m.group(0)

                novo_texto_tentativa = padrao_num.sub(replace_dose, novo_texto)
                if novo_texto_tentativa != novo_texto:
                    correcoes_virgula.append(
                        f"  Linha {idx}: {insumo} {dose_usada} → {dose_corrigida} (vírgula corrigida)"
                    )
                    novo_texto = novo_texto_tentativa

    df.at[idx, 'INSUMO_DOSE'] = novo_texto

if correcoes_virgula:
    print(f"🔧 Correções de vírgula aplicadas ({len(correcoes_virgula)}):")
    for c in correcoes_virgula:
        print(c)
else:
    print("ℹ️ Nenhuma correção de vírgula necessária.")

# ================== 1. FORÇAR SUBSTITUIÇÃO DE TODOS OS EXCESSOS ==================
for idx in range(len(df)):
    texto = str(df.at[idx, 'INSUMO_DOSE'])
    if not texto or texto == 'nan':
        continue

    doses = extrair_doses(texto)
    novo_texto = texto

    for insumo, dose_usada in doses.items():
        if insumo in dose_max and dose_usada > dose_max[insumo] + 0.001:
            padrao_sub = re.compile(
                re.escape(insumo) + r'.*?-\s*\(?[\d,]+\)?',
                re.IGNORECASE
            )
            novo_texto = padrao_sub.sub(f"{insumo} - ({dose_max[insumo]:.3f})", novo_texto)

    df.at[idx, 'INSUMO_DOSE'] = novo_texto

# ================== 2. REDISTRIBUIR O EXCEDENTE ==================
excesso_total = {}
for idx in range(len(df)):
    doses = extrair_doses(str(df.at[idx, 'INSUMO_DOSE']))
    area = float(df.at[idx, 'PRODUCAO']) if pd.notna(df.at[idx, 'PRODUCAO']) else 0.0
    for insumo, dose_usada in doses.items():
        if insumo in dose_max and dose_usada > dose_max[insumo] + 0.001:
            excesso = (dose_usada - dose_max[insumo]) * area
            excesso_total[insumo] = excesso_total.get(insumo, 0) + excesso

novas_linhas = []
for insumo, excedente in excesso_total.items():
    if excedente > 0.001:
        dose_maxima = dose_max[insumo]
        area_nova = round(excedente / dose_maxima, 4)
        nova = df.iloc[0].copy()
        nova['GLEBA'] = "REDISTRIBUIDA"
        nova['QUADRA'] = "EXTRA"
        nova['NOME_DA_FAZENDA'] = "Redistribuição Excesso"
        nova['INSUMO_DOSE'] = f"211 - {insumo} - ({dose_maxima:.3f})"
        nova['PRODUCAO'] = area_nova
        novas_linhas.append(nova)

if novas_linhas:
    df = pd.concat([df, pd.DataFrame(novas_linhas)], ignore_index=True)

df.to_excel('Capina_Corrigido.xlsx', index=False)

print("✅ Pronto!")
print(f"Excesso total redistribuído: {sum(excesso_total.values()):.3f}")
print(f"Linhas finais: {len(df)}")

In [ ]:
import pandas as pd
import re
from google.colab import files

# Upload da planilha
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]
df = pd.read_excel(nome_arquivo)

# Cálculo inicial da produção
initial_producao_sum = df['PRODUCAO'].astype(str).str.replace(',', '.').astype(float).sum()
print(f"Produção total inicial no arquivo: {initial_producao_sum:.2f}")

df['PRODUCAO'] = df['PRODUCAO'].astype(str).str.replace(',', '.').astype(float)

# Regex flexível para vírgula ou ponto
padrao = re.compile(r'(\d+)\s*-\s*(.+?)\s*-\s*\((\d+[,.]\d+)\)')

linhas = []

for idx, row in df.iterrows():
    produtos_str_list = str(row['INSUMO_DOSE']).split('/')
    products_parsed_for_current_row = 0

    for produto_str in produtos_str_list:
        produto_str_clean = produto_str.strip()
        m = padrao.search(produto_str_clean)

        if m:
            codigo_produto, nome_produto, dose_str = m.groups()
            dose = float(dose_str.replace(',', '.'))
            linhas.append({
                'ORIGINAL_ROW': idx, # Mantém a referência única da linha
                'ID': row['ID'], # Inclui a coluna ID
                'CODIGO': row['CODIGO'],
                'GLEBA': row['GLEBA'],
                'QUADRA': row['QUADRA'],
                'PRODUCAO': row['PRODUCAO'],
                'PRODUTO': f"{codigo_produto} - {nome_produto}",
                'DOSE': dose
            })
            products_parsed_for_current_row += 1

    # Se não processou nenhum produto (vazio ou hífen), mantém a linha com NA
    if products_parsed_for_current_row == 0:
        linhas.append({
            'ORIGINAL_ROW': idx,
            'ID': row['ID'], # Inclui a coluna ID
            'CODIGO': row['CODIGO'],
            'GLEBA': row['GLEBA'],
            'QUADRA': row['QUADRA'],
            'PRODUCAO': row['PRODUCAO'],
            'PRODUTO': pd.NA,
            'DOSE': pd.NA
        })

df_long = pd.DataFrame(linhas)

# Numeração sequencial de produtos por linha original
df_long['N'] = df_long.groupby('ORIGINAL_ROW').cumcount() + 1

# Pivota usando ORIGINAL_ROW no index para garantir que nenhuma linha seja mesclada/excluída
pivot_produto = df_long.pivot(index=['ORIGINAL_ROW', 'ID', 'CODIGO', 'GLEBA', 'QUADRA', 'PRODUCAO'],
                                columns='N', values='PRODUTO')
pivot_dose = df_long.pivot(index=['ORIGINAL_ROW', 'ID', 'CODIGO', 'GLEBA', 'QUADRA', 'PRODUCAO'],
                             columns='N', values='DOSE')

pivot_produto.columns = [f'PRODUTO_{i}' for i in pivot_produto.columns]
pivot_dose.columns = [f'DOSE_{i}' for i in pivot_dose.columns]

df_wide = pd.concat([pivot_produto, pivot_dose], axis=1).reset_index()

# Reordena colunas (PRODUTO_1, DOSE_1, ...)
ordem_colunas = []
max_p = pivot_produto.shape[1]
for i in range(1, max_p + 1):
    ordem_colunas.extend([f'PRODUTO_{i}', f'DOSE_{i}'])

fixed_cols = ['ID', 'CODIGO', 'GLEBA', 'QUADRA', 'PRODUCAO'] # Adiciona ID aos fixed_cols
df_wide = df_wide[fixed_cols + ordem_colunas]

# Verificação final da soma
final_producao_sum = df_wide['PRODUCAO'].sum()
print(f"\nProdução total final na planilha exportada: {final_producao_sum:.2f}")

df_wide.to_excel('planilha_final_corrigida.xlsx', index=False)
files.download('planilha_final_corrigida.xlsx')

display(df_wide.head(10))